In [6]:
import os
import sys
import pandas as pd
import numpy as np
import json
import datetime as dt
from scipy.stats import norm
import matplotlib.pyplot as plt
import quantstats
from statsmodels.tsa.stattools import adfuller

PATH = os.path.dirname(os.getcwd())
sys.path.append(PATH)
from PCA import yield_curve_decomposition
from Dates import from_excel_date, bump_date
from Plots import plot_time_series, plot_alpha_decay
from Stats import strategy_summary_statistics
from RelativeValue import FlyRV

In [2]:
tcosts = {'US': 0.0707e-4,
          'EU': 0.0434e-4,
          'Mexico': 0.2182e-4,
          'Chile': 0.3953e-4,
          'Brazil': 0.1667e-4,
          'India': 0.1260e-4,
          'China': 0.1946e-4}
fly_parameters = {
    'US': {'tenors': ['2Y', '5Y', '10Y'], 'residual_tenor': '5Y', 'window': 63, 'vol_window': 21, 'vt': 0.25, 'confidence': 0.5, 'exit_confidence': 0.7},
    'EU': {'tenors': ['2Y', '5Y', '10Y'], 'residual_tenor': '5Y', 'window': 63, 'vol_window': 21, 'vt': 0.25, 'confidence': 0.7, 'exit_confidence': 0.7},
    'Mexico': {'tenors': ['2Y', '5Y', '10Y'], 'residual_tenor': '5Y', 'window': 63, 'vol_window': 21,  'vt': 0.25, 'confidence': 0.9, 'exit_confidence': 0.5},
    'Chile': {'tenors': ['2Y', '5Y', '10Y'], 'residual_tenor': '5Y', 'window': 63, 'vol_window': 21,  'vt': 0.2, 'confidence': 0.8, 'exit_confidence': 0.5},
    'Brazil': {'tenors': ['1M', '3M', '6M'], 'residual_tenor': '5Y', 'window': 21*3, 'vol_window': 21*3,  'vt': 0.25, 'confidence': 0.6, 'exit_confidence': 0.5},
    'India': {'tenors': ['2Y', '5Y', '10Y'], 'residual_tenor': '5Y', 'window': 21*3, 'vol_window': 21,  'vt': 0.25, 'confidence': 0.8, 'exit_confidence': 0.7},
    'China': {'tenors': ['2Y', '5Y', '10Y'], 'residual_tenor': '5Y', 'window': 63, 'vol_window': 21, 'vt': 0.25, 'confidence': 0.75, 'exit_confidence': 0.5}
}

In [ ]:
index_ = pd.MultiIndex.from_tuples([('DM', 'US'),
                                    ('DM', 'EU'),
                                    ('EM LatAm', 'Mexico'),
                                    ('EM LatAm', 'Chile'),
                                    ('EM LatAm', 'Brazil'),
                                    ('EM Asia', 'India'),
                                    ('EM Asia', 'China'),])
stats = ['sharpe', 'cagr', 'volatility', 'sortino', 'calmar', 'skew', 'kurtosis',
       'max_drawdown', 'win_rate', 'daily_value_at_risk', 'breakeven']
res = pd.DataFrame(index = stats, columns = index_)
save, include_tcosts = False, False

for r, c in index_:
    print(f'Analysis for {c}...')
    fly = FlyRV(c, *fly_parameters[c]['tenors'], fly_parameters[c]['residual_tenor'])
    fly.get_residuals()
    fly.backtest(window = fly_parameters[c]['window'],
                 vol_window = fly_parameters[c]['vol_window'],
                 tc = tcosts[c] if include_tcosts else 0.0,
                 vt = fly_parameters[c]['vt'],
                 confidence = fly_parameters[c]['confidence'],
                 exit_confidence= fly_parameters[c]['exit_confidence'])
    fly.report(save=save)
    res[r, c] = fly.summary


In [3]:
# Example
country = 'US'
fly = FlyRV(country, *fly_parameters[country]['tenors'], fly_parameters[country]['residual_tenor'])
fly.get_residuals()
fly.backtest(window = fly_parameters[country]['window'],
                vol_window = fly_parameters[country]['vol_window'],
                tc = tcosts[country],
                vt = fly_parameters[country]['vt'],
                confidence = fly_parameters[country]['confidence'],
                exit_confidence= fly_parameters[country]['exit_confidence'])
fly.report(save=False)

/Users/arathreyes/Documents/Repos/AFP/AFP/RelativeValue.py:495: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.rates = pd.read_excel(PATH + f'/data/{file}.xlsx', index_col=0, sheet_name=config["Curve Name"], parse_dates=True)[tenors]


Getting PCA residuals ...
Residuals successfully computed


In [4]:
plot_time_series(fly.pca_scores, x_label='Date', y_label='Score', title=f'{fly.country} PCA Factors')

In [ ]:
# Run an ADF test on fly.pca_scores['Curvature'] to check for stationarity
result = adfuller(fly.pca_scores['Curvature'].dropna())
print('ADF Statistic: %f' % result[0])
print('p-value: %f' % result[1])    

ADF Statistic: -3.453906
p-value: 0.009253
